# 01. 데이터 불러오기 (한국도로공사 사고 + 터널)

**실행 전 준비**
1. data.go.kr에서 두 데이터 활용신청: 15164206(한국도로공사 사고 이정별 좌표 및 기상), 15081955(전국도로터널정보표준데이터)
2. 이 노트북과 같은 폴더의 `.env.example`을 `.env`로 이름 바꾸고, 마이페이지의 **일반 인증키(Decoding)** 붙여넣기
3. VS Code 오른쪽 위 **커널 선택**에서 Anaconda Python 선택 후, 위에서부터 셀 하나씩 실행(Shift+Enter)

In [ ]:
# 처음 한 번만 (이미 있으면 건너뛰어도 됨)
%pip install requests pandas -q

In [ ]:
import os, time
from pathlib import Path
import pandas as pd
import requests

ROOT = Path.cwd()                      # 이 노트북이 있는 sunsight_data 폴더
RAW_DIR = ROOT / "data" / "1_원본"
RAW_DIR.mkdir(parents=True, exist_ok=True)

BASE_URL = "https://api.odcloud.kr/api"
DATASETS = {
    "accident": "/15164206/v1/uddi:182d1361-59df-4f81-bf5e-b268d49260bf",  # 사고 _20251231, 4,458행
    "tunnel":   "/15081955/v1/uddi:9ff55d42-c4eb-4dd0-8bb1-9e4806908c88",  # 터널 _20251231
}

## 1. API 키 읽기 (.env)

In [ ]:
def get_service_key():
    # .env 파일을 먼저 읽음 (VS Code가 예전 .env 내용을 환경변수로 넣어둔 경우를 피하기 위해)
    key = None
    env_file = ROOT / ".env"
    if env_file.exists():
        for line in env_file.read_text(encoding="utf-8-sig").splitlines():
            if line.strip().startswith("DATA_GO_KR_KEY="):
                key = line.split("=", 1)[1].strip().strip('"').strip("'")
    if not key or "여기에" in key:
        key = os.environ.get("DATA_GO_KR_KEY")
    print(".env 위치:", env_file, "/ 존재:", env_file.exists())
    assert key and "여기에" not in key, ".env 파일에 일반 인증키를 넣어주세요."
    return key

SERVICE_KEY = get_service_key()
print("키 확인:", SERVICE_KEY[:5] + "...")   # 키 전체는 출력하지 않음

## 2. API 호출 함수 (페이지 반복)

In [ ]:
def fetch_page(path, page, per_page, retries=3):
    params = {"page": page, "perPage": per_page, "returnType": "JSON", "serviceKey": SERVICE_KEY}
    for attempt in range(retries):
        try:
            r = requests.get(BASE_URL + path, params=params, timeout=30)
            if r.status_code == 401:
                raise RuntimeError("401 인증 실패: 활용신청 승인 여부와 Decoding 키인지 확인 (신청 직후엔 최대 1시간 걸릴 수 있음)")
            r.raise_for_status()
            return r.json()
        except requests.RequestException as e:
            if attempt == retries - 1:
                raise
            print(f"  재시도 {attempt + 1}: {e}")
            time.sleep(2 ** attempt)

def fetch_all(path, per_page=1000):
    first = fetch_page(path, 1, per_page)
    total = first["totalCount"]
    rows = list(first["data"])
    for page in range(2, -(-total // per_page) + 1):
        rows.extend(fetch_page(path, page, per_page)["data"])
        time.sleep(0.2)
    df = pd.DataFrame(rows)
    print(f"  totalCount={total}, 받은 행={len(df)}")
    return df

## 3. 다운로드 후 원본 저장 (`data/1_원본/`)

In [ ]:
accident = fetch_all(DATASETS["accident"])
accident.to_csv(RAW_DIR / "accident.csv", index=False, encoding="utf-8-sig")

tunnel = fetch_all(DATASETS["tunnel"])
tunnel.to_csv(RAW_DIR / "tunnel.csv", index=False, encoding="utf-8-sig")

accident.head()

## 4. 원본 점검
여기서부터 출력 결과를 Claude에게 보여주면 정제 단계를 이어서 만들어요.

In [ ]:
# 다시 열 때는 API 호출 없이 저장본 사용
accident = pd.read_csv(RAW_DIR / "accident.csv", dtype=str)
print(accident.shape)
print(accident.columns.tolist())
accident.sample(5, random_state=0)

In [ ]:
print("[사고연도]"); print(accident["사고연도"].value_counts().sort_index())
t = accident["사고시간"].str.strip()
print("\n[사고시간] 길이 분포:", t.str.len().value_counts().to_dict())
print("예시:", t.sample(15, random_state=0).tolist())
print("\n[사고일자] 예시:", accident["사고일자"].head().tolist())

In [ ]:
# 33~39 범위면 위도, 124~132 범위면 경도
for c in ["X좌표", "Y좌표"]:
    v = pd.to_numeric(accident[c], errors="coerce")
    print(f"[{c}] min={v.min()}  max={v.max()}  변환실패={v.isna().sum()}")

In [ ]:
for c in ["방향", "날씨", "사고원인", "사고노선"]:
    print(f"\n[{c}] 고유값 {accident[c].nunique()}개")
    print(accident[c].value_counts().head(15).to_string())
print("\n[사고발생이정] 예시:", accident["사고발생이정"].head(10).tolist())
print("\n[결측]"); print(accident.isna().sum())

In [ ]:
# 터널: 고속도로만 몇 개인지
tunnel = pd.read_csv(RAW_DIR / "tunnel.csv", dtype=str)
print(tunnel.shape)
print(tunnel["도로종류"].value_counts())
tunnel[["터널명","도로노선명","도로노선방향","터널시작점위도","터널시작점경도","터널종료점위도","터널종료점경도","터널연장"]].head()